# 🔗 Workflow Integration - Complete SAR Processing System

Welcome to Phase 4 of the Financial Services Agentic AI Project!

In this notebook, you'll integrate both AI agents into a complete **end-to-end SAR processing workflow** that demonstrates real-world financial compliance automation.

## 🎯 Learning Objectives
- Build a complete two-stage AI workflow with human oversight
- Implement human-in-the-loop decision gates for compliance
- Generate complete SAR documents from AI analysis
- Create comprehensive audit trails for regulatory examination
- Demonstrate cost optimization through intelligent agent coordination

## 📋 Business Context
This workflow simulates how banks actually process suspicious activity reports:
1. **Risk Screening**: AI agents analyze transaction patterns for suspicious activity
2. **Human Review**: Compliance officers review AI findings before proceeding
3. **Narrative Generation**: Only approved cases get full compliance documentation
4. **SAR Filing**: Complete regulatory forms are generated for submission
5. **Audit Documentation**: Every decision is logged for regulatory examination

## 🏗️ System Architecture

```
📊 CSV Data → 🔍 Risk Analyst → 👤 Human Decision → ✅ Compliance Officer → 📄 SAR Document
              (Chain-of-Thought)    (Gate)         (ReACT Framework)     (FinCEN Ready)
```

## 🚀 Prerequisites Check

Before starting, ensure you have completed:
- ✅ Phase 1: Foundation components (`foundation_sar.py`)
- ✅ Phase 2: Risk Analyst Agent (`risk_analyst_agent.py`)
- ✅ Phase 3: Compliance Officer Agent (`compliance_officer_agent.py`)
- ✅ Both agents pass their comprehensive test scenarios

If any component is missing, return to previous notebooks to complete implementation.

In [2]:
# Setup and Environment Configuration
import os
import sys
import json
import pandas as pd
import uuid
import hashlib
from datetime import datetime, timedelta
from dotenv import load_dotenv

# Add src directory to Python path for module imports
sys.path.append(os.path.abspath('..'))

# Load environment variables
load_dotenv('../.env')

print("📚 Libraries imported successfully!")
print("🔐 Environment variables loaded")
print("📂 Source directory added to Python path")

📚 Libraries imported successfully!
🔐 Environment variables loaded
📂 Source directory added to Python path


In [3]:
# OpenAI Setup for Vocareum
import openai

# Initialize OpenAI client for Vocareum
openai_api_key = os.getenv('OPENAI_API_KEY')

if not openai_api_key:
    print("⚠️ WARNING: No OpenAI API key found!")
    print("Please set OPENAI_API_KEY in your .env file")
    print("Get your Vocareum OpenAI API key from 'Cloud Resources' in your workspace")
else:
    # Vocareum requires routing through their servers
    client = openai.OpenAI(
        base_url="https://openai.vocareum.com/v1",
        api_key=openai_api_key
    )
    print("✅ OpenAI client initialized with Vocareum routing")
    print(f"🔑 API key: {openai_api_key[:8]}...{openai_api_key[-4:]}")
    print("📍 Base URL: https://openai.vocareum.com/v1")

✅ OpenAI client initialized with Vocareum routing
🔑 API key: voc-5575...6106
📍 Base URL: https://openai.vocareum.com/v1


In [4]:
# Import Implemented Components
# Import foundation components and agents

from src.foundation_sar import (
    CustomerData,
    AccountData,
    TransactionData,
    CaseData,
    RiskAnalystOutput,
    ComplianceOfficerOutput,
    ExplainabilityLogger,
    DataLoader,
    load_csv_data
)
from src.risk_analyst_agent import RiskAnalystAgent
from src.compliance_officer_agent import ComplianceOfficerAgent

# Create agent instances
explainability_logger = ExplainabilityLogger("../outputs/audit_logs/workflow_integration.jsonl")
risk_agent = RiskAnalystAgent(client, explainability_logger)
compliance_agent = ComplianceOfficerAgent(client, explainability_logger)

print("✅ Ready to import components after implementation")

✅ Ready to import components after implementation


## 📊 Step 1: Data Loading and Preprocessing

Load the financial data and prepare it for analysis.

In [5]:
# Load and Preprocess Financial Data
# Load customer, account, and transaction data

def load_and_preprocess_data():
    """
    Load CSV data and prepare for analysis
    
    This function should:
    1. Load customers.csv, accounts.csv, transactions.csv
    2. Handle missing values appropriately
    3. Create data dictionaries for processing
    4. Return cleaned datasets
    """
    print("📊 Loading Financial Data")
    # Load CSV files from ../data/ directory
    # Handle NaN values in optional fields
    # Convert to dictionaries for processing
    
    # Example structure (uncomment and modify):
    customers_df = pd.read_csv("../data/customers.csv", dtype={'ssn_last_4': str})
    accounts_df = pd.read_csv("../data/accounts.csv")
    transactions_df = pd.read_csv("../data/transactions.csv")
    
    # Handle NaN values
    transactions_df['counterparty'] = transactions_df['counterparty'].fillna('')
    transactions_df['location'] = transactions_df['location'].fillna('')
    customers_df['phone'] = customers_df['phone'].fillna('')
    
    # Convert to dictionaries
    customers_data = customers_df.to_dict('records')
    accounts_data = accounts_df.to_dict('records')
    transactions_data = transactions_df.to_dict('records')
    
    print(f"📈 Loaded: {len(customers_data)} customers, {len(accounts_data)} accounts, {len(transactions_data)} transactions")
    return customers_data, accounts_data, transactions_data

# Load data
customers_data, accounts_data, transactions_data = load_and_preprocess_data()

📊 Loading Financial Data
📈 Loaded: 150 customers, 178 accounts, 4268 transactions


## 🎯 Step 2: Customer Risk Screening

Implement intelligent customer screening to identify high-risk cases for detailed analysis.

In [6]:
# Implement Customer Risk Screening
# Create risk-based customer screening logic

def screen_high_risk_customers(customers_data, accounts_data, transactions_data, top_n=5):
    """
    Implement risk-based customer screening
    
    Screening criteria include:
    1. High risk ratings (Medium, High)
    2. Large transaction amounts (>$100K total)
    3. High transaction frequency (>50 transactions)
    4. Recent activity patterns
    
    Returns top N highest-risk customers for detailed analysis
    """
    print("🔍 Customer Risk Screening")
    # Implement risk-based screening criteria
    # Calculate risk scores for each customer
    # Select top N customers for SAR analysis
    
    # Example screening logic (uncomment and modify):
    selected_customers = []
    
    for customer in customers_data:
        # Get customer accounts and transactions
        customer_accounts = [acc for acc in accounts_data if acc['customer_id'] == customer['customer_id']]
        customer_transactions = [txn for txn in transactions_data if any(txn['account_id'] == acc['account_id'] for acc in customer_accounts)]
        
        # Calculate risk indicators
        total_amount = sum(abs(txn['amount']) for txn in customer_transactions)
        transaction_count = len(customer_transactions)
        risk_rating = customer['risk_rating']
        
        # Apply screening criteria
        risk_flags = []
        if risk_rating in ['Medium', 'High']:
            risk_flags.append('high_risk_rating')
        if total_amount > 100000:
            risk_flags.append('large_amounts')
        if transaction_count > 50:
            risk_flags.append('high_frequency')
        
        # Select high-risk customers
        if len(risk_flags) >= 2:  # Multiple risk flags
            selected_customers.append({
                'customer': customer,
                'accounts': customer_accounts,
                'transactions': customer_transactions,
                'total_amount': total_amount,
                'transaction_count': transaction_count,
                'risk_flags': risk_flags
            })
    # 
    # # Sort by risk score and take top N
    selected_customers.sort(key=lambda x: (len(x['risk_flags']), x['total_amount']), reverse=True)
    return selected_customers[:top_n]

# Run customer screening
selected_customers = screen_high_risk_customers(customers_data, accounts_data, transactions_data)

🔍 Customer Risk Screening


## 🤖 Step 3: Two-Stage AI Analysis with Human Gates

Implement the core two-stage workflow:
1. **Stage 1**: Risk Analyst performs Chain-of-Thought analysis
2. **Human Gate**: Review and decision to proceed
3. **Stage 2**: Compliance Officer generates ReACT narratives (only if approved)

In [7]:
# Implement Two-Stage AI Workflow
# Build the complete workflow with human decision gates

def run_two_stage_sar_workflow(selected_customers):
    """
    Implement complete two-stage SAR processing workflow
    
    For each customer:
    1. Create CaseData object
    2. Run Risk Analyst analysis (Chain-of-Thought)
    3. Present findings to human reviewer
    4. Get human decision (proceed/reject)
    5. If approved: Run Compliance Officer (ReACT)
    6. Generate complete SAR document
    7. Log all decisions for audit
    """
    print("🤖 Two-Stage SAR Processing Workflow")
    # Implement complete workflow logic
    
    # Initialize tracking
    processed_cases = []
    approved_sars = []
    rejected_cases = []
    audit_decisions = []
    
    # For each selected customer:
    # 1. Create CaseData from customer, accounts, transactions
    # 2. Run Risk Analyst analysis
    # 3. Display analysis results to human reviewer
    # 4. Get human decision (input('Proceed with SAR filing? (yes/no): '))
    # 5. If 'yes': Run Compliance Officer narrative generation
    # 6. Create complete SAR document with all metadata
    # 7. Save SAR to ../outputs/filed_sars/ directory
    # 8. Log decision to audit trail
    
    # Example workflow structure (uncomment and implement):
    for i, customer_data in enumerate(selected_customers, 1):
        print(f"\n🔍 CUSTOMER {i}/{len(selected_customers)}: {customer_data['customer']['name']}")
        print("=" * 60)
        start_time = datetime.now()
        try:
            # Create case data
            loader = DataLoader(explainability_logger)
            case_data = loader.create_case_from_data(
                customer_data['customer'],
                customer_data['accounts'], 
                customer_data['transactions']
            )
        except Exception as e:
            print(f"❌ Error processing customer: {customer_data['customer']}")
            processed_cases.append({
            'case_id': customer_data['customer']['customer_id'],
            'stage': 'case_constrction',
            'outcome': 'failed',
            'error': str(e)})
            continue
        try:
            # STAGE 1: Risk Analysis
            print("🔍 STAGE 1: Risk Analysis")
            risk_analysis = risk_agent.analyze_case(case_data)

            # Display analysis results
            print(f"Classification: {risk_analysis.classification}")
            print(f"Confidence: {risk_analysis.confidence_score:.2f}")
            print(f"Risk Level: {risk_analysis.risk_level}")
            print(f"Reasoning: {risk_analysis.reasoning}")

        except Exception as e:
            print(f"❌ Error processing customer: {e}")
            processed_cases.append({
            'case_id': case_data.case_id,
            'stage': 'risk_analysis',
            'outcome': 'failed',
            'error': str(e)})
            continue
            
        # HUMAN DECISION GATE
        decision = input("🤔 Proceed with SAR filing? (yes/no): ").strip().lower()
        should_proceed = decision in ['yes', 'y']
        identity = input("What is your identity?")
        rationale = input("What is your rationale?")
        execution_time_ms = (datetime.now() - start_time).total_seconds() * 1000
        explainability_logger.log_agent_action(identity, "human_decision", case_data.case_id, 
                    str(case_data), should_proceed, rationale, 
                    execution_time_ms, True 
                    )
        audit_decisions.append({
            'case_id': case_data.case_id,
            'customer_name': case_data.customer.name,
            'decision': 'PROCEED' if should_proceed else 'REJECT',
            'ai_classification': risk_analysis.classification,
            'ai_confidence': risk_analysis.confidence_score,
            'reviewer_decision': decision
        })
        if should_proceed:
            try:
                # STAGE 2: Compliance Narrative
                print("📝 STAGE 2: Compliance Narrative Generation")
                compliance_review = compliance_agent.generate_compliance_narrative(case_data, risk_analysis)
                
                # Generate complete SAR document
                sar_document = create_sar_document(case_data, risk_analysis, compliance_review, identity)
                
                # Save SAR
                save_sar_document(sar_document)
                approved_sars.append(sar_document)
                
                print(f"✅ SAR FILED: {sar_document['sar_metadata']['sar_id']}")
            except Exception as e:
                print(f"❌ Error processing customer: {e}")
                
                processed_cases.append({
                'case_id': case_data.case_id,
                'stage': 'compliance_narrative',
                'outcome': 'failed',
                'error': str(e)})
                continue
        else:
            rejected_cases.append({'case_id': case_data.case_id, 'reason': 'human_rejection'})
            print("❌ SAR REJECTED by human reviewer")
            
        # Log decision
        processed_cases.append({
            'case_id': case_data.case_id,
            'stage': 'compliance_narrative',
            'outcome': 'successful' if should_proceed else 'REJECT',
            'error': None})
        
    
    return processed_cases, approved_sars, rejected_cases, audit_decisions

# Run the complete workflow
processed_cases, approved_sars, rejected_cases, audit_decisions = run_two_stage_sar_workflow(selected_customers)



🤖 Two-Stage SAR Processing Workflow

🔍 CUSTOMER 1/5: Jacqueline Rodriguez
🔍 STAGE 1: Risk Analysis
Classification: Money_Laundering
Confidence: 0.85
Risk Level: High
Reasoning: The customer has a high volume of wire transfers, including a significant incoming wire transfer of $76,517.25 and multiple outgoing wire transfers totaling $49,517.51 within a short time frame. This pattern, combined with the customer's high-risk rating and the nature of the transactions, suggests potential money laundering activities aimed at obscuring the source of funds.
📝 STAGE 2: Compliance Narrative Generation
Customer identification: Jacqueline Rodriguez, Suspicious activity description and Transaction amounts and dates: Total incoming amount: $76,517.25 on 2025-07-29; Total outgoing amount: $49,517.51 on 2025-07-30 and 2025-07-31; Total deposits amount: $0.00 on None, Location of the transaction: None, Why activity is suspicious: The significant incoming wire transfer of $76,517.25 followed by multiple 

## 📄 Step 4: SAR Document Generation

Create complete, FinCEN-ready SAR documents with all required metadata.

In [8]:
# Implement SAR Document Generation
# Create complete SAR documents for regulatory submission

def create_sar_document(case_data, risk_analysis, compliance_review, identity):
    """
    Create complete SAR document
    
    SAR document should include:
    1. SAR metadata (ID, filing date, type, checksum)
    2. Subject information (customer details)
    3. Suspicious activity description
    4. AI analysis results
    5. Compliance narrative
    6. Regulatory citations
    7. Filing institution information
    """
    print("📄 Creating SAR Document")
    # Generate unique SAR ID
    # Include all required metadata
    # Format for FinCEN submission
    
    # Example SAR document structure (uncomment and implement):
    sar_id = f"SAR_{uuid.uuid4()}"
    filing_date = datetime.now().isoformat()
    
    sar_document = {
        'sar_metadata': {
            'sar_id': sar_id,
            'filing_date': filing_date,
            'filing_type': 'Suspicious Activity Report',
            'ai_generated': True,
            'review_status': 'human_approved'
        },
        'subject_information': {
            'customer_name': case_data.customer.name,
            'customer_id': case_data.customer.customer_id,
            'address': case_data.customer.address,
            'customer_since': case_data.customer.customer_since,
            'risk_rating': case_data.customer.risk_rating
        },
        'suspicious_activity': {
            'classification': risk_analysis.classification,
            'risk_level': risk_analysis.risk_level,
            'confidence_score': risk_analysis.confidence_score,
            'narrative': compliance_review.narrative,
            'key_indicators': risk_analysis.key_indicators,
            'ai_reasoning': risk_analysis.reasoning
        },
        'regulatory_compliance': {
            'citations': getattr(compliance_review, 'regulatory_citations', []),
            'narrative_word_count': len(compliance_review.narrative.split()),
            'compliance_status': 'approved' if compliance_review.completeness_check else 'not approved'
        },
        'audit_trail': {
            'case_id': case_data.case_id,
            'processing_date': filing_date,
            'ai_agents_used': ['RiskAnalyst', 'ComplianceOfficer'],
            'human_reviewer': identity
        }
    }
    
    return sar_document

def save_sar_document(sar_document):
    """Save SAR document to outputs directory"""
    # Save SAR to ../outputs/filed_sars/ directory
    os.makedirs("../outputs/filed_sars", exist_ok=True)
    filename = f"../outputs/filed_sars/{sar_document['sar_metadata']['sar_id']}.json"
    with open(filename, 'w') as f:
        json.dump(sar_document, f, indent=2)

print("📄 SAR document generation functions defined")

📄 SAR document generation functions defined


## 📊 Step 5: Workflow Metrics and Analysis

Analyze the efficiency and effectiveness of your AI-powered SAR processing system.

In [9]:
# Implement Workflow Analysis and Metrics
# Calculate efficiency metrics and cost analysis

def analyze_workflow_efficiency(processed_cases, approved_sars, rejected_cases, audit_decisions):
    """
    Calculate workflow efficiency metrics
    
    Metrics to calculate:
    1. Processing efficiency (time per case)
    2. Cost optimization (two-stage vs single-stage)
    3. Human decision patterns
    4. AI accuracy validation
    5. Regulatory compliance rates
    """
    # Workflow Efficiency Analysis
    # Calculate processing metrics
    
    # Example metrics calculation:
    total_cases = len(processed_cases)
    approved_cases = len(approved_sars)
    rejected_cases_count = len(rejected_cases)
    
    if total_cases > 0:
        approval_rate = approved_cases / total_cases
        rejection_rate = rejected_cases_count / total_cases
    else:
        approval_rate = rejection_rate = 0
    
    print(f"📈 WORKFLOW METRICS:")
    print(f"   Total Cases Processed: {total_cases}")
    print(f"   SARs Filed: {approved_cases}")
    print(f"   Cases Rejected: {rejected_cases_count}")
    print(f"   Approval Rate: {approval_rate:.1%}")
    print(f"   Rejection Rate: {rejection_rate:.1%}")
    
    # Cost optimization analysis
    print(f"\n💰 COST OPTIMIZATION:")
    print(f"   Two-stage processing saves costs by only running")
    print(f"   expensive compliance generation on approved cases")
    print(f"   Cost savings: {rejection_rate:.1%} of compliance calls avoided")
    
    print("💡 Implement metrics calculation after running workflow")

def validate_ai_decisions(audit_decisions):
    """Analyze AI decision patterns and accuracy"""
    # Validate AI classification accuracy
    # Analyze confidence score distributions
    # Review human override patterns
    print("The accuracy of AI classification is decent. The confidence scores are all 0.85. Human overrides cases where the evidence or reasoning is not concrete.")

# Run analysis
analyze_workflow_efficiency(processed_cases, approved_sars, rejected_cases, audit_decisions)
validate_ai_decisions(audit_decisions)

📈 WORKFLOW METRICS:
   Total Cases Processed: 5
   SARs Filed: 4
   Cases Rejected: 1
   Approval Rate: 80.0%
   Rejection Rate: 20.0%

💰 COST OPTIMIZATION:
   Two-stage processing saves costs by only running
   expensive compliance generation on approved cases
   Cost savings: 20.0% of compliance calls avoided
💡 Implement metrics calculation after running workflow
The accuracy of AI classification is decent. The confidence scores are all 0.85. Human overrides cases where the evidence or reasoning is not concrete.


## 🏁 Step 6: Complete System Demonstration

Test your complete system with comprehensive scenarios to validate production readiness.

In [10]:
# Run Complete System Test
# Demonstrate your complete SAR processing system

def demonstrate_complete_system():
    """
    Run complete system demonstration
    
    This should:
    1. Process multiple customers through the complete workflow
    2. Show both approved and rejected cases
    3. Generate multiple SAR documents
    4. Demonstrate audit trail creation
    5. Show efficiency metrics
    """
    print("🏁 Complete SAR Processing System Demonstration")
    # Run complete workflow with multiple customers
    # Show both approval and rejection scenarios
    # Generate audit reports
    # Calculate final efficiency metrics
    
    # Example demonstration (uncomment after implementation):
    print("🚀 Running complete system test...")
    
    # Load fresh data
    customers_data, accounts_data, transactions_data = load_and_preprocess_data()
    
    # Screen customers
    selected_customers = screen_high_risk_customers(customers_data, accounts_data, transactions_data, top_n=3)
    
    # Run workflow
    processed_cases, approved_sars, rejected_cases, audit_decisions = run_two_stage_sar_workflow(selected_customers)
    
    # Generate final report
    analyze_workflow_efficiency(processed_cases, approved_sars, rejected_cases, audit_decisions)
    
    print(f"🎉 System demonstration complete!")
    print(f"📄 SAR documents saved to: ../outputs/filed_sars/")
    print(f"📊 Audit logs saved to: ../outputs/audit_logs/")

demonstrate_complete_system()

🏁 Complete SAR Processing System Demonstration
🚀 Running complete system test...
📊 Loading Financial Data
📈 Loaded: 150 customers, 178 accounts, 4268 transactions
🔍 Customer Risk Screening
🤖 Two-Stage SAR Processing Workflow

🔍 CUSTOMER 1/3: Jacqueline Rodriguez
🔍 STAGE 1: Risk Analysis
Classification: Money_Laundering
Confidence: 0.85
Risk Level: High
Reasoning: The financial activity shows a significant volume of wire transfers, particularly a large incoming wire transfer of 76,517.25, followed by multiple outgoing wire transfers totaling 49,517.51 within a short time frame. This pattern is indicative of potential money laundering activities, as it involves large amounts being moved quickly, which is a common tactic to obscure the source of funds.
📝 STAGE 2: Compliance Narrative Generation
Customer identification: Jacqueline Rodriguez, Suspicious activity description and Transaction amounts and dates: Total incoming amount: $76,517.25 on 2025-07-29; Total outgoing amount: $49,517.51 

## 📝 Implementation Checklist

### ✅ Workflow Integration Deliverables
- [ ] **Data Loading**: Load and preprocess CSV data with proper error handling
- [ ] **Customer Screening**: Implement risk-based screening to identify high-risk cases
- [ ] **Two-Stage Workflow**: Build complete Risk Analyst → Human Gate → Compliance Officer flow
- [ ] **Human Decision Gates**: Implement interactive approval/rejection points
- [ ] **SAR Document Generation**: Create complete FinCEN-ready documents with metadata
- [ ] **Audit Trail Creation**: Log all decisions and reasoning for regulatory examination
- [ ] **Efficiency Metrics**: Calculate cost optimization and processing efficiency
- [ ] **System Demonstration**: Test complete workflow with multiple scenarios

### ✅ Testing and Validation Requirements
- [ ] **Component Validation**: Verify all foundation components and agents are available
- [ ] **Integration Testing**: Run comprehensive test suites for all components with proper sys.path setup
- [ ] **End-to-End Testing**: Test complete workflow with automated scenarios
- [ ] **Error Handling Testing**: Validate graceful handling of edge cases and failures
- [ ] **Output Validation**: Ensure SAR documents meet regulatory standards
- [ ] **Performance Testing**: Measure workflow efficiency and processing times

### ✅ Technical Requirements
- [ ] **Error Handling**: Robust exception handling for all workflow steps
- [ ] **Data Validation**: Proper validation of all inputs and outputs
- [ ] **File Management**: Organize outputs in appropriate directories
- [ ] **Logging**: Comprehensive audit logging for compliance
- [ ] **Performance**: Efficient processing of multiple cases
- [ ] **User Experience**: Clear prompts and feedback for human reviewers
- [ ] **Test Infrastructure**: Proper test imports and sys.path configuration

### ✅ Business Requirements  
- [ ] **Regulatory Compliance**: Ensure all SAR documents meet FinCEN requirements
- [ ] **Cost Optimization**: Demonstrate savings from two-stage processing
- [ ] **Audit Readiness**: Create examination-ready documentation
- [ ] **Quality Assurance**: Validate AI decisions with human oversight
- [ ] **Scalability**: Design for processing larger datasets
- [ ] **Production Readiness**: Complete testing validates system reliability

## 🎯 Success Criteria

By completion, your integrated system should:
- ✅ Process real financial data with proper validation
- ✅ Execute complete two-stage AI workflow with human gates
- ✅ Generate regulatory-compliant SAR documents
- ✅ Create comprehensive audit trails for all decisions
- ✅ Demonstrate measurable cost optimization benefits
- ✅ Handle errors gracefully and provide clear user feedback
- ✅ Pass all integration and end-to-end tests
- ✅ Meet production-ready quality standards

## 🚀 Next Steps

1. **Complete Implementation**: Fill in all TODO sections with working code
2. **Run Integration Tests**: Validate all components work together properly
3. **Execute End-to-End Tests**: Test complete workflow with automated scenarios
4. **Test Thoroughly**: Run complete workflow with various manual scenarios
5. **Validate Outputs**: Ensure SAR documents meet regulatory requirements
6. **Document Results**: Create final project documentation and metrics
7. **Prepare Presentation**: Demonstrate your system's capabilities and business value

**Congratulations on building a complete AI-powered SAR processing system! 🎉**

## 🧪 Step 7: Workflow Testing and Validation

Before finalizing your implementation, validate your complete system with comprehensive testing.

In [11]:
# 🧪 Workflow Integration Testing
# Validate your complete system with integration tests

import sys
import os

# Add tests directory to Python path for importing test modules
project_root = os.path.abspath('..')
tests_path = os.path.join(project_root, 'tests')
if tests_path not in sys.path:
    sys.path.insert(0, tests_path)

print(f"📁 Added tests directory to Python path: {tests_path}")

def run_integration_tests():
    """
    Run comprehensive integration tests to validate the complete workflow
    
    Tests include:
    1. Foundation components integration
    2. Agent communication and data flow
    3. End-to-end workflow execution
    4. Error handling and edge cases
    5. Output validation and compliance
    """
    print("🧪 Comprehensive Integration Testing")
    
    try:
        # Import all test modules
        from test_foundation import TestCustomerData, TestAccountData, TestTransactionData, TestCaseData
        from test_risk_analyst import TestRiskAnalystAgent
        from test_compliance_officer import TestComplianceOfficerAgent
        import pytest
        
        print("🔍 Running Foundation Component Tests...")
        foundation_result = pytest.main([
            f"{tests_path}/test_foundation.py", 
            "-v", 
            "--tb=short"
        ])
        
        print("🔍 Running Risk Analyst Agent Tests...")
        risk_result = pytest.main([
            f"{tests_path}/test_risk_analyst.py", 
            "-v", 
            "--tb=short"
        ])
        
        print("📝 Running Compliance Officer Agent Tests...")
        compliance_result = pytest.main([
            f"{tests_path}/test_compliance_officer.py", 
            "-v", 
            "--tb=short"
        ])
        
        # Calculate overall test results
        all_passed = foundation_result == 0 and risk_result == 0 and compliance_result == 0
        
        print("\n" + "="*60)
        print("📊 INTEGRATION TEST RESULTS:")
        print(f"   Foundation Components: {'✅ PASS' if foundation_result == 0 else '❌ FAIL'}")
        print(f"   Risk Analyst Agent: {'✅ PASS' if risk_result == 0 else '❌ FAIL'}")
        print(f"   Compliance Officer Agent: {'✅ PASS' if compliance_result == 0 else '❌ FAIL'}")
        print(f"   Overall Status: {'✅ ALL TESTS PASSED' if all_passed else '❌ SOME TESTS FAILED'}")
        
        if all_passed:
            print("\n🎉 Your system is ready for production workflow testing!")
            print("📝 Proceed to run the complete system demonstration.")
        else:
            print("\n⚠️ Fix failing tests before running the complete workflow.")
            print("📝 Return to previous notebooks to fix component issues.")
        
        return all_passed
            
    except ImportError as e:
        print(f"❌ Import Error: {e}")
        print("💡 Make sure all components are implemented:")
        print("   • foundation_sar.py")
        print("   • risk_analyst_agent.py") 
        print("   • compliance_officer_agent.py")
        return False

def validate_workflow_components():
    """Validate that all required components are available for integration"""
    print("🔍 Validating Workflow Components")
    
    components_status = {
        'foundation_sar': False,
        'risk_analyst_agent': False,
        'compliance_officer_agent': False,
        'test_modules': False
    }
    
    try:
        # Check foundation components
        from src.foundation_sar import CustomerData, CaseData, ExplainabilityLogger, DataLoader
        components_status['foundation_sar'] = True
        print("✅ Foundation components available")
    except ImportError:
        print("❌ Foundation components not available")
    
    try:
        # Check risk analyst agent
        from src.risk_analyst_agent import RiskAnalystAgent
        components_status['risk_analyst_agent'] = True
        print("✅ Risk Analyst Agent available")
    except ImportError:
        print("❌ Risk Analyst Agent not available")
    
    try:
        # Check compliance officer agent
        from src.compliance_officer_agent import ComplianceOfficerAgent
        components_status['compliance_officer_agent'] = True
        print("✅ Compliance Officer Agent available")
    except ImportError:
        print("❌ Compliance Officer Agent not available")
    
    try:
        # Check test modules
        from test_foundation import TestCustomerData
        from test_risk_analyst import TestRiskAnalystAgent  
        from test_compliance_officer import TestComplianceOfficerAgent
        components_status['test_modules'] = True
        print("✅ Test modules available")
    except ImportError:
        print("❌ Test modules not available")
    
    all_ready = all(components_status.values())
    
    print(f"\n📊 Component Status: {'✅ ALL READY' if all_ready else '⚠️ INCOMPLETE'}")
    if not all_ready:
        print("💡 Complete missing components before running integration tests")
    
    return all_ready

# Run component validation
components_ready = validate_workflow_components()

# Run integration tests if components are ready
if components_ready:
    print("\n🚀 All components ready - you can run integration tests!")
    run_integration_tests()
else:
    print("\n📋 Complete component implementation first, then run integration tests")

📁 Added tests directory to Python path: /workspace/cd14685-fin-serv-agentic-c1-classroom/project/starter/tests
🔍 Validating Workflow Components
✅ Foundation components available
✅ Risk Analyst Agent available
✅ Compliance Officer Agent available
⚠️  Foundation components not yet implemented - tests will be skipped
💡 Implement the classes in src/foundation_sar.py to run these tests
✅ Test modules available

📊 Component Status: ✅ ALL READY

🚀 All components ready - you can run integration tests!
🧪 Comprehensive Integration Testing
🔍 Running Foundation Component Tests...
============================= test session starts ==============================
platform linux -- Python 3.13.0, pytest-8.4.1, pluggy-1.6.0 -- /opt/venv/bin/python
cachedir: .pytest_cache
rootdir: /workspace/cd14685-fin-serv-agentic-c1-classroom/project/starter
plugins: anyio-4.10.0
collecting ... collected 10 items

../tests/test_foundation.py::TestCustomerData::test_valid_customer_data SKIPPED [ 10%]
../tests/test_foun

In [12]:
# 🎯 End-to-End Workflow Testing
# Test the complete workflow with known test scenarios

def test_complete_workflow():
    """
    Test the complete SAR processing workflow end-to-end
    
    This test should:
    1. Load test data (can use a subset of actual data)
    2. Run customer screening
    3. Execute two-stage AI analysis
    4. Simulate human decisions (automated for testing)
    5. Generate SAR documents
    6. Validate all outputs
    7. Check audit trail completeness
    """
    print("🎯 End-to-End Workflow Testing")
    
    # Example end-to-end test structure (uncomment when ready):
    try:
        print("🚀 Starting end-to-end workflow test...")
        
        # Test data preparation
        print("📊 Loading test data...")
        customers_data, accounts_data, transactions_data = load_and_preprocess_data()
        
        if not customers_data:
            print("⚠️ No test data available - implement data loading first")
            return False
        
        # Test customer screening
        print("🔍 Testing customer screening...")
        selected_customers = screen_high_risk_customers(
            customers_data, accounts_data, transactions_data, top_n=2
        )
        
        if not selected_customers:
            print("⚠️ No customers selected - check screening criteria")
            return False
        
        print(f"✅ Selected {len(selected_customers)} customers for testing")
        
        # Test workflow with automated decisions (for testing)
        print("🤖 Testing automated workflow...")
        test_results = {
            'cases_processed': 0,
            'sars_generated': 0,
            'errors': []
        }
        
        for customer_data in selected_customers:
            try:
                # Create case
                loader = DataLoader(explainability_logger)
                case_data = loader.create_case_from_data(
                    customer_data['customer'],
                    customer_data['accounts'],
                    customer_data['transactions']
                )
                
                # Test Risk Analyst
                risk_analysis = risk_agent.analyze_case(case_data)
                assert hasattr(risk_analysis, 'classification'), "Risk analysis missing classification"
                assert hasattr(risk_analysis, 'confidence_score'), "Risk analysis missing confidence"
                
                # Test Compliance Officer (simulate approval)
                compliance_review = compliance_agent.generate_compliance_narrative(case_data, risk_analysis)
                assert hasattr(compliance_review, 'narrative'), "Compliance review missing narrative"
                
                # Test SAR generation
                sar_document = create_sar_document(case_data, risk_analysis, compliance_review, 'human')
                assert sar_document, "SAR document generation failed"
                
                test_results['cases_processed'] += 1
                test_results['sars_generated'] += 1
                
                print(f"✅ Successfully processed: {customer_data['customer']['name']}")
                
            except Exception as e:
                test_results['errors'].append(f"Error processing {customer_data['customer']['name']}: {e}")
                print(f"❌ Error processing: {customer_data['customer']['name']}: {e}")
        
        # Test results summary
        print("\n📊 END-TO-END TEST RESULTS:")
        print(f"   Cases Processed: {test_results['cases_processed']}")
        print(f"   SARs Generated: {test_results['sars_generated']}")
        print(f"   Errors: {len(test_results['errors'])}")
        
        if test_results['errors']:
            print("❌ Test Errors:")
            for error in test_results['errors']:
                print(f"     • {error}")
        
        success = len(test_results['errors']) == 0 and test_results['cases_processed'] > 0
        
        if success:
            print("\n🎉 END-TO-END TEST PASSED!")
            print("✅ Your complete workflow is ready for production use!")
        else:
            print("\n⚠️ END-TO-END TEST HAD ISSUES")
            print("📝 Fix the errors above before deploying to production")
        
        return success
        
    except Exception as e:
        print(f"❌ End-to-end test failed: {e}")
        print("💡 Ensure all components are properly implemented")
        return False

# Run end-to-end test
test_success = test_complete_workflow()

🎯 End-to-End Workflow Testing
🚀 Starting end-to-end workflow test...
📊 Loading test data...
📊 Loading Financial Data
📈 Loaded: 150 customers, 178 accounts, 4268 transactions
🔍 Testing customer screening...
🔍 Customer Risk Screening
✅ Selected 2 customers for testing
🤖 Testing automated workflow...
Customer identification: Jacqueline Rodriguez, Suspicious activity description and Transaction amounts and dates: Total incoming amount: $76,517.25 on 2025-07-29; Total outgoing amount: $49,517.51 on 2025-07-30 and 2025-07-31. Location of the transaction: None, Why activity is suspicious: The pattern of a large incoming wire transfer followed by multiple significant outgoing wire transfers raises suspicion of potential money laundering, as the funds are moved quickly without clear legitimate purposes.
📄 Creating SAR Document
✅ Successfully processed: Jacqueline Rodriguez
Customer identification: Michael Stanley, Suspicious activity description and Transaction amounts and dates: Total incoming